# Haiku를 sub-agent로 사용하기

이 recipe에서는 Claude 3 Haiku sub-agent 모델을 사용하여 실적 발표 PDF에서 관련 정보를 추출하고 Apple의 2023년 재무 실적 보고서를 분석하는 방법을 보여줍니다. 그런 다음 Claude 3 Opus를 사용하여 질문에 대한 응답을 생성하고 matplotlib으로 응답에 함께 제공할 그래프를 만듭니다.

## 1단계: 환경 설정
먼저 필요한 라이브러리를 설치하고 Claude API client를 설정하겠습니다.

In [ ]:
%pip install anthropic IPython PyMuPDF matplotlib

In [89]:
# 필요한 라이브러리 import
import base64
import io
import os
from concurrent.futures import ThreadPoolExecutor

import fitz
import requests
from anthropic import Anthropic
from PIL import Image

# Claude API client 설정
client = Anthropic()
MODEL_NAME = "claude-haiku-4-5"

## 2단계: 문서 수집 및 질문하기
이 예제에서는 Apple의 2023 회계연도 재무제표를 모두 사용하고 해당 연도의 순매출에 대해 질문하겠습니다.

In [90]:
# Apple 실적 발표 PDF URL 목록
pdf_urls = [
    "https://www.apple.com/newsroom/pdfs/fy2023-q4/FY23_Q4_Consolidated_Financial_Statements.pdf",
    "https://www.apple.com/newsroom/pdfs/fy2023-q3/FY23_Q3_Consolidated_Financial_Statements.pdf",
    "https://www.apple.com/newsroom/pdfs/FY23_Q2_Consolidated_Financial_Statements.pdf",
    "https://www.apple.com/newsroom/pdfs/FY23_Q1_Consolidated_Financial_Statements.pdf",
]

# 사용자 질문
QUESTION = "How did Apple's net sales change quarter to quarter in the 2023 financial year and what were the key contributors to the changes?"

## 3단계: PDF 다운로드 및 이미지로 변환
다음으로 실적 발표 PDF를 다운로드하고 base64로 인코딩된 PNG 이미지로 변환하는 함수를 정의하겠습니다. 이 PDF에는 기존 PDF parser로 parsing하기 어려운 표가 가득하므로 이 작업이 필요합니다. 이미지로 변환하여 Haiku에 전달하는 편이 더 쉽습니다.

```download_pdf``` 함수는 주어진 URL에서 PDF 파일을 다운로드하고 지정된 폴더에 저장합니다. ```pdf_to_base64_pngs``` 함수는 PDF를 base64로 인코딩된 PNG 이미지 목록으로 변환합니다.

In [ ]:
# URL에서 PDF 파일을 다운로드하여 지정된 폴더에 저장하는 함수
def download_pdf(url, folder):
    response = requests.get(url, timeout=60)
    if response.status_code == 200:
        file_name = os.path.join(folder, url.split("/")[-1])
        with open(file_name, "wb") as file:
            file.write(response.content)
        return file_name
    else:
        print(f"Failed to download PDF from {url}")
        return None


# PDF를 base64로 인코딩된 PNG 이미지 목록으로 변환하는 함수 정의
def pdf_to_base64_pngs(pdf_path, quality=75, max_size=(1024, 1024)):
    # PDF 파일 열기
    doc = fitz.open(pdf_path)

    base64_encoded_pngs = []

    # PDF의 각 페이지 순회
    for page_num in range(doc.page_count):
        # 페이지 로드
        page = doc.load_page(page_num)

        # 페이지를 PNG 이미지로 렌더링
        pix = page.get_pixmap(matrix=fitz.Matrix(300 / 72, 300 / 72))

        # Pixmap을 PIL Image로 변환
        image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # 최대 크기를 초과하면 이미지 크기 조정
        if image.size[0] > max_size[0] or image.size[1] > max_size[1]:
            image.thumbnail(max_size, Image.Resampling.LANCZOS)

        # PIL Image를 base64로 인코딩된 PNG로 변환
        image_data = io.BytesIO()
        image.save(image_data, format="PNG", optimize=True, quality=quality)
        image_data.seek(0)
        base64_encoded = base64.b64encode(image_data.getvalue()).decode("utf-8")
        base64_encoded_pngs.append(base64_encoded)

    # PDF 문서 닫기
    doc.close()

    return base64_encoded_pngs


# 다운로드한 PDF를 저장할 폴더
folder = "../images/using_sub_agents"


# 디렉터리가 없으면 생성
os.makedirs(folder)

# PDF 동시 다운로드
with ThreadPoolExecutor() as executor:
    pdf_paths = list(executor.map(download_pdf, pdf_urls, [folder] * len(pdf_urls)))

# pdf_paths에서 None 값(다운로드 실패) 모두 제거
pdf_paths = [path for path in pdf_paths if path is not None]

ThreadPoolExecutor를 사용하여 PDF를 동시에 다운로드하고 파일 경로를 pdf_paths에 저장합니다.

## 4단계: Opus를 사용하여 Haiku용 구체적인 prompt 생성
Opus를 orchestrator로 사용하고 사용자가 제공한 질문에 따라 각 Haiku sub-agent용 구체적인 prompt를 작성하도록 하겠습니다.

In [102]:
def generate_haiku_prompt(question):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"Based on the following question, please generate a specific prompt for an LLM sub-agent to extract relevant information from an earning's report PDF. Each sub-agent only has access to a single quarter's earnings report. Output only the prompt and nothing else.\n\nQuestion: {question}",
                }
            ],
        }
    ]

    response = client.messages.create(model="claude-opus-4-8", max_tokens=2048, messages=messages)

    return response.content[0].text


haiku_prompt = generate_haiku_prompt(QUESTION)
print(haiku_prompt)

Extract the following information from the Apple earnings report PDF for the quarter:
1. Apple's net sales for the quarter
2. Quarter-over-quarter change in net sales
3. Key product categories, services, or regions that contributed significantly to the change in net sales
4. Any explanations provided for the changes in net sales

Organize the extracted information in a clear, concise format focusing on the key data points and insights related to the change in net sales for the quarter.


## 5단계: PDF에서 정보 추출
이제 질문을 정의하고 sub-agent Haiku 모델을 사용하여 PDF에서 정보를 추출하겠습니다. 각 모델에서 얻은 정보를 명확하게 정의된 XML tag 집합으로 정리합니다.

In [107]:
def extract_info(pdf_path, haiku_prompt):
    base64_encoded_pngs = pdf_to_base64_pngs(pdf_path)

    messages = [
        {
            "role": "user",
            "content": [
                *[
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": base64_encoded_png,
                        },
                    }
                    for base64_encoded_png in base64_encoded_pngs
                ],
                {"type": "text", "text": haiku_prompt},
            ],
        }
    ]

    response = client.messages.create(model="claude-haiku-4-5", max_tokens=2048, messages=messages)

    return response.content[0].text, pdf_path


def process_pdf(pdf_path):
    return extract_info(pdf_path, haiku_prompt)


# Haiku sub-agent 모델로 PDF 동시 처리
with ThreadPoolExecutor() as executor:
    extracted_info_list = list(executor.map(process_pdf, pdf_paths))

extracted_info = ""
# 각 모델 호출에서 추출한 정보 표시
for info in extracted_info_list:
    extracted_info += (
        '<info quarter="' + info[1].split("/")[-1].split("_")[1] + '">' + info[0] + "</info>\n"
    )
print(extracted_info)

<info quarter="Q4">According to the condensed consolidated statements of operations, Apple's net sales changed as follows in the 2023 financial year:

Quarter Ended September 30, 2023:
- Total net sales were $89,498 million, up from $90,146 million in the prior year quarter.
- Product sales were $67,184 million and services sales were $22,314 million.

Key contributors to the changes:
- Product sales decreased from $70,958 million in the prior year quarter.
- Services sales increased from $19,188 million in the prior year quarter.

Overall, Apple's total net sales decreased slightly compared to the same quarter in the prior year, driven by a decline in product sales which was partially offset by growth in services sales.</info>
<info quarter="Q3">Based on the financial statements provided, Apple's net sales changed as follows between the quarters in the 2023 financial year:

- Net sales increased from $81,797 million in the three months ended July 1, 2023 to $82,959 million in the thre

Sub-agent 모델을 사용하여 PDF에서 정보를 동시에 추출하고 추출된 정보를 결합합니다. 그런 다음 질문과 추출한 정보를 포함하여 강력한 모델에 전달할 message를 준비하고 응답과 matplotlib 코드를 생성하도록 요청합니다.

## 6단계: Opus에 정보를 전달하여 응답 생성
Sub-agent를 사용하여 각 PDF에서 정보를 가져왔으므로 이제 Opus를 호출하여 실제로 질문에 답하고 답변에 함께 제공할 그래프를 만드는 코드를 작성하도록 하겠습니다.

In [108]:
# 강력한 모델용 message 준비
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"Based on the following extracted information from Apple's earnings releases, please provide a response to the question: {QUESTION}\n\nAlso, please generate Python code using the matplotlib library to accompany your response. Enclose the code within <code> tags.\n\nExtracted Information:\n{extracted_info}",
            }
        ],
    }
]

# 강력한 모델로 matplotlib 코드 생성
response = client.messages.create(model="claude-opus-4-8", max_tokens=4096, messages=messages)

generated_response = response.content[0].text
print("Generated Response:")
print(generated_response)

Generated Response:
Based on the extracted information from Apple's earnings releases, Apple's net sales changed as follows in the 2023 financial year:

In Q1, net sales increased from $117,154 million in the previous quarter to $123,945 million, driven by increases in both product sales and services revenue.

In Q2, net sales decreased by around $2,442 million compared to the prior six-month period, primarily due to a decrease in product sales, which was partially offset by an increase in services sales.

In Q3, net sales increased by approximately $1,162 million compared to the previous quarter, with growth in both product sales and services sales contributing to the overall increase.

In Q4, total net sales decreased slightly compared to the same quarter in the prior year, driven by a decline in product sales, which was partially offset by growth in services sales.

Here's a Python code snippet using the matplotlib library to visualize the quarterly net sales data:

<code>
import ma

## 7단계: 응답 추출 및 Matplotlib 코드 실행
마지막으로 생성된 응답에서 matplotlib 코드를 추출하고 실행하여 매출 성장 추세를 시각화하겠습니다.

생성된 응답에서 matplotlib 코드와 코드가 아닌 응답을 추출하는 ```extract_code_and_response``` 함수를 정의합니다. 코드가 아닌 응답을 출력하고 matplotlib 코드가 있으면 실행합니다.

Sandbox 외부에서 모델이 작성한 코드에 ```exec```를 사용하는 것은 좋은 관행이 아니지만, 이 데모의 목적을 위해 여기서는 사용한다는 점에 유의하세요 :)

In [ ]:
# 응답에서 matplotlib 코드 추출
# 응답에서 코드 및 코드가 아닌 부분을 추출하는 함수
def extract_code_and_response(response):
    start_tag = "<code>"
    end_tag = "</code>"
    start_index = response.find(start_tag)
    end_index = response.find(end_tag)
    if start_index != -1 and end_index != -1:
        code = response[start_index + len(start_tag) : end_index].strip()
        non_code_response = response[:start_index].strip()
        return code, non_code_response
    else:
        return None, response.strip()


matplotlib_code, non_code_response = extract_code_and_response(generated_response)

print(non_code_response)
if matplotlib_code:
    # 추출한 matplotlib 코드 실행
    # 참고: 여기서는 모델이 생성한 시각화 코드를 실행하는 데모 목적으로 exec를 사용합니다.
    # Production에서는 신뢰할 수 없는 코드를 실행할 때 sandbox 환경을 사용하세요.
    exec(matplotlib_code)  # noqa: S102
else:
    print("No matplotlib code found in the response.")